## IMPORT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import sklearn
from datetime import datetime
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import joblib
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform
import pandas as pd

## Modèle 1 : Prédiction de la taille d'un panier pour un acheteur d'alcool

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

# Récupère si l'article dans le panier est de l'alcool
df["is_alcohol"] = df["department"].str.contains("alcohol", case=False, na=False).astype(int)

# Regroupe toutes les lignes qui on le même "order_id" ensuite compte le nombre de lignes dans chaque groupe ( donc le nombre de produit ) et transforme le résultat en dataframe
basket_size = df.groupby("order_id").size().reset_index(name="basket_size")

orders = df.groupby("order_id").agg(
    user_id = ("user_id", "first"),
    order_number = ("order_number", "first"),
    order_hour_of_day = ("order_hour_of_day", "first"),
    order_dow = ("order_dow", "first"), 
    days_since_prior_order = ("days_since_prior_order", "first"),
    alcohol_count = ("is_alcohol", "sum"),
    reorder_count = ("reordered", "sum"),
    number_different_departments = ("department", "nunique"),
    number_different_aisles = ("aisle", "nunique"),
).reset_index()

orders = orders.merge(basket_size, on="order_id")

# Ici on récupère si c'est le weekend ( classique burger ) 
orders["is_weekend"] = orders["order_dow"].isin([0, 6]).astype(int)
# Ici on récupère le moment de la journée (oui j'ai remis le 24 parceque sinon ça prenait pas en compte 22 et 23h singe) 
orders["time_of_day"] = pd.cut(
    orders["order_hour_of_day"],
    bins=[0, 6, 12, 17, 21, 24],
    labels=["nuit", "matin", "apres_midi", "soir", "soiree"],
    right=False
).astype(str)

# On fait le tri par utilisateur et numéro de commande
orders_sorted = orders.sort_values(["user_id", "order_number"])

# Ça en résumé : Je fais une moyenne ligne par ligne et à chaque ligne avec le ".expanding" je fais la moyenne en récupérant la ligne d'avant aussi ex : commande 1 = 5 la moyenne c'est 5, commande 2 = [8 ET 5] donc la moyenne c'est 6.5 etc ...
orders_sorted["user_avg_basket"] = (
    orders_sorted.groupby("user_id")["basket_size"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Taille du panier précedent
orders_sorted["user_last_basket"] = (
    orders_sorted.groupby("user_id")["basket_size"]
    .transform(lambda x: x.shift(1))
)

# Nombre de commandes d'avant
orders_sorted["user_order_count"] = orders_sorted["order_number"] - 1

# Moyenne de rachat d'avant
orders_sorted["user_avg_reorder"] = (
    orders_sorted.groupby("user_id")["reorder_count"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Remplacer les Na par la médiane globale
for col in ["user_avg_basket", "user_last_basket", "user_avg_reorder"]:
    orders_sorted[col] = orders_sorted[col].fillna(orders_sorted[col].median())

orders = orders_sorted.copy()

#/--------------------------------------------------------------\#
#|------------------------ Entraînement ------------------------|#
#\--------------------------------------------------------------/#

feature_cols = [
    "order_hour_of_day", "order_dow",
    "is_weekend", "time_of_day",
    "days_since_prior_order", "order_number",
    "alcohol_count", "reorder_count",
    "number_different_departments", "number_different_aisles",
    "user_avg_basket", "user_last_basket",
    "user_order_count", "user_avg_reorder",
]

X = pd.get_dummies(orders[feature_cols])
y = orders["basket_size"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42
)

model = HistGradientBoostingRegressor(
    max_iter=440,
    learning_rate=np.float64(0.026183714048995665),
    max_leaf_nodes=82,
    min_samples_leaf=13,
    l2_regularization=np.float64(0.012709563372047594),
    random_state=150,
    max_depth=9
)
# Lance l'entraînements du modèle
model.fit(X_train, y_train)

#/--------------------------------------------------------------\#
#|------------------------- Évaluation -------------------------|#
#\--------------------------------------------------------------/#

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE :{mae}")
print(f"RMSE : {rmse}")
print(f"R2 : {r2}")

#/---------------------------------------------------------------\#
#|------------------------ Justification ------------------------|#
#\---------------------------------------------------------------/#

# param_dist = {
#     "max_iter": randint(200, 800),
#     "learning_rate": uniform(0.01, 0.15),
#     "max_leaf_nodes": randint(31, 512),
#     "min_samples_leaf": randint(10, 100),
#     "l2_regularization": uniform(0.0, 0.5),
#     "max_depth": randint(3, 12),
#     "random_state": randint(0, 200)
# }

# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# search = RandomizedSearchCV(
#     estimator = HistGradientBoostingRegressor(),
#     param_distributions = param_dist,
#     n_iter = 30,
#     scoring = "r2",
#     cv = cv,
#     n_jobs = -1,
#     verbose = 2,
#     random_state= 42,
#     return_train_score = True
# )

# search.fit(X_train, y_train)

# print(search.best_params_)
# print(f"Meilleur R² (CV) : {search.best_score_:.4f}")

# best_model = search.best_estimator_
# y_pred = best_model.predict(X_test)
# r2_2 = r2_score(y_test, y_pred)
# print(f"R² final (test) : {r2_2}")

MAE :1.3271381401047326
RMSE : 4.131955381317046
R2 : 0.9256455171714738


## Modèle 2 : Prédiction du contenu du panier

In [ ]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

top_500_products = df["product_name"].value_counts().head(500).index
df_filtered = df[df["product_name"].isin(top_500_products)]



grouped_by_orders = df_filtered.groupby("order_id").agg({
    "product_name": list,
    "user_id": "first",
    "order_number": "first",
    "order_dow": "first",
    "order_hour_of_day": "first",
    "days_since_prior_order": "first",
    
}).reset_index()

grouped_by_orders["size_of_the_basket"] = grouped_by_orders["product_name"].map(len)
print(grouped_by_orders)

feature_cols = [
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "size_of_the_basket"
]

X = pd.get_dummies(grouped_by_orders[feature_cols])

y = grouped_by_orders["product_name"]

mlb = MultiLabelBinarizer(sparse_output=True)
y_matrice = mlb.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_matrice, test_size=0.2, random_state=42
)

model = OneVsRestClassifier(
    BernoulliNB()
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(y_pred)

## Modèle 3 : Prédire

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")
df = df.dropna()

orders = df.groupby("order_id").agg(
    user_id                = ("user_id",               "first"),
    order_number           = ("order_number",           "first"),
    order_hour_of_day      = ("order_hour_of_day",      "first"),
    order_dow              = ("order_dow",              "first"),
    days_since_prior_order = ("days_since_prior_order", "first"),
).reset_index()

orders = orders.sort_values(["user_id", "order_number"]).reset_index(drop=True)

orders["is_weekend"] = orders["order_dow"].isin([0, 6]).astype(int)
orders["time_of_day"] = pd.cut(
    orders["order_hour_of_day"],
    bins=[0, 6, 12, 17, 21, 24],
    labels=["nuit", "matin", "apres_midi", "soir", "soiree"],
    right=False
).astype(str)
orders['user_order_count'] = orders.groupby('user_id').cumcount()
orders["user_avg_days"] = orders.groupby("user_id")["days_since_prior_order"].transform(
    lambda x: x.shift(1).expanding().mean()
)
orders["user_std_days"] = orders.groupby("user_id")["days_since_prior_order"].transform(
    lambda x: x.shift(1).expanding().std()
)

orders = orders.dropna()
next_days_since_order = orders['days_since_prior_order'].shift(-1)
tagets = ["is_weekend","time_of_day","order_hour_of_day","order_dow","user_order_count","user_avg_days","user_std_days"]

X2 = pd.get_dummies(orders[tagets])
y2 = orders.groupby('user_id')['days_since_prior_order'].shift(-1)

mask = y2.notna()
X2 = X2[mask]
y2 = y2[mask]

model = HistGradientBoostingRegressor(
    max_iter=661,                 
    learning_rate=np.float64(0.10563362070328196),
    max_leaf_nodes=161,            
    min_samples_leaf=10,                    
    random_state=4,
    l2_regularization= np.float64(0.36480308916903204),
    max_depth=5
)

X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test , y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(mae)
print(rmse)
print(r2)

#/---------------------------------------------------------------\#
#|------------------------ Justification ------------------------|#
#\---------------------------------------------------------------/#

# param_dist = {
#     "max_iter": randint(200, 800),
#     "learning_rate": uniform(0.01, 0.15),
#     "max_leaf_nodes": randint(31, 512),
#     "min_samples_leaf": randint(10, 100),
#     "l2_regularization": uniform(0.0, 0.5),
#     "max_depth": randint(3, 12),
#     "random_state": randint(0, 200)
# }

# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# search = RandomizedSearchCV(
#     estimator = HistGradientBoostingRegressor(),
#     param_distributions = param_dist,
#     n_iter = 30,
#     scoring = "r2",
#     cv = cv,
#     n_jobs = -1,
#     verbose = 2,
#     random_state= 42,
#     return_train_score = True
# )

# search.fit(X_train, y_train)

# print(search.best_params_)
# print(f"Meilleur R² (CV) : {search.best_score_:.4f}")

# best_model = search.best_estimator_
# y_pred = best_model.predict(X_test)
# r2_2 = r2_score(y_test, y_pred)
# print(f"R² final (test) : {r2_2}")


Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END l2_regularization=0.023332831606807714, learning_rate=0.15606332782621887, max_depth=5, max_iter=766, max_leaf_nodes=274, min_samples_leaf=73, random_state=130; total time=   3.7s
[CV] END l2_regularization=0.36099938613341237, learning_rate=0.15078290635236252, max_depth=4, max_iter=391, max_leaf_nodes=474, min_samples_leaf=30, random_state=160; total time=   3.8s
[CV] END l2_regularization=0.023332831606807714, learning_rate=0.15606332782621887, max_depth=5, max_iter=766, max_leaf_nodes=274, min_samples_leaf=73, random_state=130; total time=   4.0s
[CV] END l2_regularization=0.36099938613341237, learning_rate=0.15078290635236252, max_depth=4, max_iter=391, max_leaf_nodes=474, min_samples_leaf=30, random_state=160; total time=   4.2s
[CV] END l2_regularization=0.023332831606807714, learning_rate=0.15606332782621887, max_depth=5, max_iter=766, max_leaf_nodes=274, min_samples_leaf=73, random_state=130; total time=   